# Notebook 06 — Inferencia estadística pareada

## Grain AI–MDPI

Este notebook realiza el análisis inferencial confirmatorio sobre las predicciones congeladas de los notebooks 04 y 05.

### Objetivos

1. Calcular intervalos de confianza mediante **bootstrap estratificado por clase y agrupado por `duplicate_group_id`**.
2. Comparar de forma pareada las representaciones:
   - R1 frente a R3.
   - R1 frente a R4.
   - R3 frente a R4.
3. Aplicar la prueba exacta de **McNemar** a los errores pareados.
4. Cuantificar el efecto de la calibración por temperatura en:
   - Log-loss.
   - Brier score multiclase.
5. Aplicar corrección de **Holm** por comparaciones múltiples.
6. Conservar los campeones seleccionados exclusivamente mediante validación cruzada en entrenamiento.
7. Exportar tablas, figuras, protocolo, manifiesto de integridad y un ZIP final.

### Restricciones metodológicas

- No se entrenan modelos nuevos.
- No se reajustan hiperparámetros.
- No se selecciona un nuevo campeón con la prueba.
- Las comparaciones están predefinidas antes del análisis.
- Las inferencias se limitan a la partición de prueba congelada.
- El análisis evalúa datos morfométricos tabulares, no robustez de imágenes.

In [ ]:
# ============================================================
# 1. Dependencias y entorno
# ============================================================
import importlib
import subprocess
import sys

REQUIRED_PACKAGES = {
    "pyarrow": "pyarrow",
    "statsmodels": "statsmodels",
}

for module_name, package_name in REQUIRED_PACKAGES.items():
    if importlib.util.find_spec(module_name) is None:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", package_name]
        )

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import platform
import shutil
import zipfile
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from scipy.stats import binomtest
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef,
    log_loss,
)
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore")

print("Python:", sys.version.split()[0])
print("CPU disponibles:", os.cpu_count())
print("Plataforma:", platform.platform())

## Configuración

`N_BOOTSTRAP = 2000` es el valor recomendado para las tablas finales.  
Para una prueba rápida del flujo puede cambiarse temporalmente a 300–500, pero los resultados destinados al manuscrito deben generarse con 2000 o más réplicas.

In [ ]:
# ============================================================
# 2. Configuración reproducible
# ============================================================
RANDOM_SEED = 20260803
N_BOOTSTRAP = 2000
CONFIDENCE_LEVEL = 0.95
ALPHA = 1.0 - CONFIDENCE_LEVEL

DATASETS = ["dry_bean", "iniap_rice"]
REPRESENTATIONS = [
    "R1_full_native",
    "R2_measured_core",
    "R3_invariant_core",
    "R4_hybrid_log_area",
]
ALGORITHMS = [
    "multinomial_logistic_regression",
    "rbf_svm",
    "extra_trees",
    "catboost",
]

# Contrastes confirmatorios definidos antes de abrir los resultados inferenciales.
REPRESENTATION_CONTRASTS = [
    ("R1_full_native", "R3_invariant_core", "R1_minus_R3"),
    ("R1_full_native", "R4_hybrid_log_area", "R1_minus_R4"),
    ("R3_invariant_core", "R4_hybrid_log_area", "R3_minus_R4"),
]

DISCRIMINATION_METRICS = [
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "mcc",
]
PROBABILITY_METRICS = [
    "log_loss",
    "brier_multiclass",
]

LOCAL_ROOT = Path("/content/Grain_AI_MDPI_notebook06")
INPUT_DIR = LOCAL_ROOT / "00_input"
EXTRACT04_DIR = LOCAL_ROOT / "01_extracted_notebook04"
EXTRACT05_DIR = LOCAL_ROOT / "02_extracted_notebook05"
OUTPUT_ROOT = LOCAL_ROOT / "03_outputs"

PROTOCOL_DIR = OUTPUT_ROOT / "03_protocol"
RESULTS_DIR = OUTPUT_ROOT / "07_results" / "06_PAIRED_INFERENCE"
FIGURES_DIR = OUTPUT_ROOT / "08_figures" / "06_PAIRED_INFERENCE"
TABLES_DIR = OUTPUT_ROOT / "09_tables" / "06_PAIRED_INFERENCE"

for directory in [
    INPUT_DIR,
    EXTRACT04_DIR,
    EXTRACT05_DIR,
    PROTOCOL_DIR,
    RESULTS_DIR,
    FIGURES_DIR,
    TABLES_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Réplicas bootstrap:", N_BOOTSTRAP)
print("Nivel de confianza:", CONFIDENCE_LEVEL)
print("Semilla:", RANDOM_SEED)

In [ ]:
# ============================================================
# 3. Montaje de Google Drive y localización de entradas
# ============================================================
try:
    from google.colab import drive, files
    drive.mount("/content/drive", force_remount=False)
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    files = None
    print("No se detectó Google Colab. Se usarán rutas locales.")

DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/Grain_Robustness_Q1")
DRIVE_PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

ZIP04_NAME = "NOTEBOOK04_MODEL_SELECTION_OUTPUTS.zip"
ZIP05_NAME = "NOTEBOOK05_BASELINE_CALIBRATION_OUTPUTS.zip"

def find_exact_file(root: Path, filename: str):
    if not root.exists():
        return None
    matches = sorted(root.rglob(filename))
    if not matches:
        return None
    # Si hay más de una copia, usar la modificada más recientemente.
    return max(matches, key=lambda path: path.stat().st_mtime)

def obtain_input_zip(filename: str) -> Path:
    found = find_exact_file(DRIVE_PROJECT_ROOT, filename)
    if found is not None:
        print(f"Encontrado en Drive: {found}")
        return found

    local_candidate = Path("/content") / filename
    if local_candidate.exists():
        print(f"Encontrado en /content: {local_candidate}")
        return local_candidate

    if IN_COLAB and files is not None:
        print(f"No se encontró {filename}. Selecciónelo para subirlo.")
        uploaded = files.upload()
        if filename not in uploaded:
            raise FileNotFoundError(
                f"Debe subir exactamente el archivo {filename}."
            )
        destination = INPUT_DIR / filename
        destination.write_bytes(uploaded[filename])
        return destination

    raise FileNotFoundError(
        f"No se encontró {filename} en {DRIVE_PROJECT_ROOT}."
    )

zip04_path = obtain_input_zip(ZIP04_NAME)
zip05_path = obtain_input_zip(ZIP05_NAME)

print("ZIP04:", zip04_path)
print("ZIP05:", zip05_path)

In [ ]:
# ============================================================
# 4. Extracción limpia y localización de archivos requeridos
# ============================================================
def clean_extract(zip_path: Path, destination: Path):
    if destination.exists():
        shutil.rmtree(destination)
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as archive:
        archive.extractall(destination)

clean_extract(zip04_path, EXTRACT04_DIR)
clean_extract(zip05_path, EXTRACT05_DIR)

SELECTED_CONFIG_FILE = (
    EXTRACT04_DIR
    / "07_results"
    / "04_MODEL_SELECTION"
    / "03_selected_model_configurations.csv"
)
GLOBAL_METRICS_FILE = (
    EXTRACT05_DIR
    / "07_results"
    / "05_BASELINE_CALIBRATION"
    / "01_baseline_test_global_metrics.csv"
)
PREDICTION_ROOT = (
    EXTRACT05_DIR
    / "06_predictions"
    / "05_BASELINE"
)

required_paths = [
    SELECTED_CONFIG_FILE,
    GLOBAL_METRICS_FILE,
    PREDICTION_ROOT,
]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Faltan archivos requeridos después de extraer:\n"
        + "\n".join(missing)
    )

selected_configurations = pd.read_csv(SELECTED_CONFIG_FILE)
reported_global_metrics = pd.read_csv(GLOBAL_METRICS_FILE)

print("Configuraciones seleccionadas:", len(selected_configurations))
print("Filas de métricas reportadas:", len(reported_global_metrics))
print("Raíz de predicciones:", PREDICTION_ROOT)

## Carga y auditoría de predicciones

Las predicciones se ordenan por `record_id`. Para cada conjunto se verifica que todas las representaciones y algoritmos contengan exactamente las mismas observaciones, etiquetas y grupos.

In [ ]:
# ============================================================
# 5. Carga de las 32 matrices de predicción de prueba
# ============================================================
def load_class_map(path: Path):
    mapping = json.loads(path.read_text(encoding="utf-8"))
    return [mapping[str(index)] for index in range(len(mapping))]

def load_prediction(dataset_id, representation, algorithm):
    directory = PREDICTION_ROOT / dataset_id / representation
    prediction_path = directory / f"{algorithm}_test_predictions.parquet"
    class_map_path = directory / f"{algorithm}_class_map.json"

    if not prediction_path.exists():
        raise FileNotFoundError(prediction_path)
    if not class_map_path.exists():
        raise FileNotFoundError(class_map_path)

    frame = pd.read_parquet(prediction_path)
    class_order = load_class_map(class_map_path)

    required_columns = {
        "record_id",
        "duplicate_group_id",
        "class_label",
        "partition",
        "predicted_native",
        "predicted_calibrated",
        "correct_native",
        "correct_calibrated",
    }
    missing_columns = sorted(required_columns - set(frame.columns))
    if missing_columns:
        raise ValueError(
            f"Columnas faltantes en {prediction_path}: {missing_columns}"
        )

    native_probability_columns = [
        f"prob_native_{index}" for index in range(len(class_order))
    ]
    calibrated_probability_columns = [
        f"prob_calibrated_{index}" for index in range(len(class_order))
    ]

    for column in native_probability_columns + calibrated_probability_columns:
        if column not in frame.columns:
            raise ValueError(f"Falta la columna {column} en {prediction_path}")

    frame = frame.copy()
    frame["record_id"] = frame["record_id"].astype(str)
    frame["duplicate_group_id"] = frame["duplicate_group_id"].astype(str)
    frame["class_label"] = frame["class_label"].astype(str)
    frame = frame.sort_values("record_id").reset_index(drop=True)

    if not (frame["partition"] == "test").all():
        raise ValueError(f"{prediction_path} contiene filas fuera de test.")

    for prefix in ["native", "calibrated"]:
        columns = [
            f"prob_{prefix}_{index}" for index in range(len(class_order))
        ]
        probabilities = frame[columns].to_numpy(dtype=float)
        if not np.isfinite(probabilities).all():
            raise ValueError(f"Probabilidades no finitas: {prediction_path}")
        if not np.allclose(probabilities.sum(axis=1), 1.0, atol=1e-8):
            raise ValueError(
                f"Las probabilidades no suman uno: {prediction_path}"
            )

    return {
        "frame": frame,
        "class_order": class_order,
        "native_probability_columns": native_probability_columns,
        "calibrated_probability_columns": calibrated_probability_columns,
        "prediction_path": prediction_path,
    }

predictions = {}
for dataset_id in DATASETS:
    for representation in REPRESENTATIONS:
        for algorithm in ALGORITHMS:
            key = (dataset_id, representation, algorithm)
            predictions[key] = load_prediction(*key)

print("Modelos cargados:", len(predictions))

In [ ]:
# ============================================================
# 6. Validación de alineación, grupos y campeones de entrenamiento
# ============================================================
alignment_rows = []

for dataset_id in DATASETS:
    reference_key = (
        dataset_id,
        REPRESENTATIONS[0],
        ALGORITHMS[0],
    )
    reference = predictions[reference_key]["frame"][
        ["record_id", "duplicate_group_id", "class_label"]
    ].copy()

    for representation in REPRESENTATIONS:
        for algorithm in ALGORITHMS:
            key = (dataset_id, representation, algorithm)
            current = predictions[key]["frame"][
                ["record_id", "duplicate_group_id", "class_label"]
            ].copy()

            aligned = reference.equals(current)
            alignment_rows.append(
                {
                    "dataset_id": dataset_id,
                    "representation": representation,
                    "algorithm": algorithm,
                    "rows": len(current),
                    "aligned_with_reference": aligned,
                }
            )

alignment_audit = pd.DataFrame(alignment_rows)
if not alignment_audit["aligned_with_reference"].all():
    display(alignment_audit[~alignment_audit["aligned_with_reference"]])
    raise RuntimeError("Las predicciones no están perfectamente alineadas.")

# Verificar que cada grupo pertenezca a una sola clase.
group_checks = []
for dataset_id in DATASETS:
    base = predictions[
        (dataset_id, REPRESENTATIONS[0], ALGORITHMS[0])
    ]["frame"]
    group_label_counts = (
        base.groupby("duplicate_group_id")["class_label"].nunique()
    )
    group_checks.append(
        {
            "dataset_id": dataset_id,
            "rows": len(base),
            "groups": base["duplicate_group_id"].nunique(),
            "classes": base["class_label"].nunique(),
            "groups_with_multiple_labels": int(
                (group_label_counts > 1).sum()
            ),
        }
    )

group_audit = pd.DataFrame(group_checks)
if (group_audit["groups_with_multiple_labels"] > 0).any():
    raise RuntimeError("Hay grupos asociados con más de una clase.")

# Campeones seleccionados únicamente mediante CV en entrenamiento.
training_champions = (
    selected_configurations.sort_values(
        ["dataset_id", "macro_f1_mean", "log_loss_mean", "candidate_id"],
        ascending=[True, False, True, True],
    )
    .groupby("dataset_id", as_index=False)
    .head(1)
    .reset_index(drop=True)
)

display(group_audit)
display(training_champions[
    [
        "dataset_id",
        "representation",
        "algorithm",
        "candidate_id",
        "macro_f1_mean",
        "macro_f1_sd",
        "log_loss_mean",
    ]
])

## Funciones estadísticas

El bootstrap se realiza en dos niveles:

1. Estratificación por clase.
2. Remuestreo con reemplazo de grupos completos dentro de cada clase.

De esta forma, los duplicados exactos de Dry Bean no se separan durante el remuestreo.

In [ ]:
# ============================================================
# 7. Funciones rápidas de métricas y bootstrap agrupado
# ============================================================
def encode_true_labels(frame, class_order):
    class_to_index = {
        class_label: index
        for index, class_label in enumerate(class_order)
    }
    y_true = frame["class_label"].astype(str).to_numpy()
    return np.array(
        [class_to_index[label] for label in y_true],
        dtype=np.int64,
    )


def log_loss_terms_for_policy(
    probabilities,
    y_true_indices,
    policy,
):
    """
    Reproduce distintas políticas históricas de sklearn.log_loss.

    La política se detecta automáticamente comparando contra las métricas
    congeladas del notebook 05. Esto evita depender de la versión de
    scikit-learn disponible en Colab.
    """
    probabilities = np.asarray(probabilities, dtype=float)

    if policy.startswith("machine_epsilon"):
        epsilon = np.finfo(probabilities.dtype).eps
    elif policy.startswith("fixed_1e-15"):
        epsilon = 1e-15
    else:
        raise ValueError(f"Política de log-loss desconocida: {policy}")

    clipped = np.clip(
        probabilities,
        epsilon,
        1.0 - epsilon,
    )

    if policy.endswith("_normalized"):
        clipped = clipped / clipped.sum(
            axis=1,
            keepdims=True,
        )
    elif not policy.endswith("_raw"):
        raise ValueError(f"Política de log-loss inválida: {policy}")

    selected_probability = clipped[
        np.arange(len(y_true_indices)),
        y_true_indices,
    ]
    return -np.log(selected_probability)


LOG_LOSS_POLICY_CANDIDATES = [
    "machine_epsilon_normalized",
    "machine_epsilon_raw",
    "fixed_1e-15_normalized",
    "fixed_1e-15_raw",
]


def prepare_metric_arrays(
    frame,
    probabilities,
    class_order,
    log_loss_policy,
):
    y_indices = encode_true_labels(frame, class_order)
    predicted_indices = probabilities.argmax(axis=1).astype(np.int64)

    selected_probability = probabilities[
        np.arange(len(y_indices)),
        y_indices,
    ]

    # Brier multiclase:
    # ||p - y||² = sum(p²) - 2*p_clase_verdadera + 1
    brier_terms = (
        np.sum(probabilities ** 2, axis=1)
        - 2.0 * selected_probability
        + 1.0
    )

    return {
        "y_indices": y_indices,
        "predicted_indices": predicted_indices,
        "log_loss_terms": log_loss_terms_for_policy(
            probabilities,
            y_indices,
            log_loss_policy,
        ),
        "brier_terms": brier_terms,
        "n_classes": len(class_order),
    }


def fast_metrics(metric_arrays, sampled_indices=None):
    if sampled_indices is None:
        y_indices = metric_arrays["y_indices"]
        predicted_indices = metric_arrays["predicted_indices"]
        log_loss_terms = metric_arrays["log_loss_terms"]
        brier_terms = metric_arrays["brier_terms"]
    else:
        y_indices = metric_arrays["y_indices"][sampled_indices]
        predicted_indices = metric_arrays[
            "predicted_indices"
        ][sampled_indices]
        log_loss_terms = metric_arrays[
            "log_loss_terms"
        ][sampled_indices]
        brier_terms = metric_arrays[
            "brier_terms"
        ][sampled_indices]

    n_classes = metric_arrays["n_classes"]
    matrix = np.bincount(
        y_indices * n_classes + predicted_indices,
        minlength=n_classes * n_classes,
    ).reshape(n_classes, n_classes).astype(float)

    true_counts = matrix.sum(axis=1)
    predicted_counts = matrix.sum(axis=0)
    true_positive = np.diag(matrix)
    total = matrix.sum()

    accuracy = float(true_positive.sum() / total)

    valid_classes = true_counts > 0
    recall = np.divide(
        true_positive,
        true_counts,
        out=np.zeros_like(true_positive),
        where=valid_classes,
    )
    balanced_accuracy = float(recall[valid_classes].mean())

    f1_denominator = true_counts + predicted_counts
    class_f1 = np.divide(
        2.0 * true_positive,
        f1_denominator,
        out=np.zeros_like(true_positive),
        where=f1_denominator > 0,
    )
    macro_f1 = float(class_f1.mean())

    correct_total = true_positive.sum()
    numerator = (
        correct_total * total
        - np.dot(true_counts, predicted_counts)
    )
    denominator_left = (
        total ** 2 - np.dot(predicted_counts, predicted_counts)
    )
    denominator_right = (
        total ** 2 - np.dot(true_counts, true_counts)
    )
    denominator = np.sqrt(
        max(0.0, denominator_left)
        * max(0.0, denominator_right)
    )
    mcc = float(numerator / denominator) if denominator > 0 else 0.0

    return {
        "accuracy": accuracy,
        "balanced_accuracy": balanced_accuracy,
        "macro_f1": macro_f1,
        "mcc": mcc,
        "log_loss": float(np.mean(log_loss_terms)),
        "brier_multiclass": float(np.mean(brier_terms)),
    }


def detect_log_loss_policy(
    predictions,
    reported_global_metrics,
):
    """
    Selecciona la convención de clipping/normalización que reproduce mejor
    el log-loss congelado del notebook 05.

    No optimiza modelos ni utiliza etiquetas para selección predictiva:
    únicamente asegura compatibilidad numérica entre versiones de sklearn.
    """
    audit_rows = []

    for policy in LOG_LOSS_POLICY_CANDIDATES:
        differences = []

        for _, reported in reported_global_metrics.iterrows():
            key = (
                reported["dataset_id"],
                reported["representation"],
                reported["algorithm"],
            )
            status = reported["probability_status"]
            bundle = predictions[key]
            frame = bundle["frame"]
            class_order = bundle["class_order"]
            probabilities = probability_matrix(bundle, status)
            y_indices = encode_true_labels(frame, class_order)

            recalculated = float(
                log_loss_terms_for_policy(
                    probabilities,
                    y_indices,
                    policy,
                ).mean()
            )
            reported_value = float(reported["log_loss"])
            difference = recalculated - reported_value
            differences.append(abs(difference))

            audit_rows.append(
                {
                    "policy": policy,
                    "dataset_id": key[0],
                    "representation": key[1],
                    "algorithm": key[2],
                    "probability_status": status,
                    "reported_log_loss": reported_value,
                    "recalculated_log_loss": recalculated,
                    "difference": difference,
                    "absolute_difference": abs(difference),
                }
            )

    detailed_audit = pd.DataFrame(audit_rows)
    policy_summary = (
        detailed_audit.groupby("policy", as_index=False)
        .agg(
            mean_absolute_difference=(
                "absolute_difference",
                "mean",
            ),
            median_absolute_difference=(
                "absolute_difference",
                "median",
            ),
            max_absolute_difference=(
                "absolute_difference",
                "max",
            ),
        )
        .sort_values(
            [
                "max_absolute_difference",
                "mean_absolute_difference",
                "policy",
            ]
        )
        .reset_index(drop=True)
    )

    selected_policy = str(policy_summary.iloc[0]["policy"])
    return selected_policy, policy_summary, detailed_audit


def build_group_bootstrap_sampler(base_frame):
    group_to_indices = {
        str(group_id): indices.to_numpy(dtype=int)
        for group_id, indices in base_frame.groupby(
            "duplicate_group_id",
            sort=False,
        ).groups.items()
    }

    group_metadata = (
        base_frame.groupby("duplicate_group_id", sort=False)
        .agg(
            class_label=("class_label", "first"),
            class_count=("class_label", "nunique"),
        )
        .reset_index()
    )
    group_metadata["duplicate_group_id"] = (
        group_metadata["duplicate_group_id"].astype(str)
    )

    if (group_metadata["class_count"] != 1).any():
        raise ValueError("Un grupo contiene más de una clase.")

    groups_by_class = {
        class_label: subset["duplicate_group_id"].to_numpy()
        for class_label, subset in group_metadata.groupby(
            "class_label",
            sort=True,
        )
    }

    def sample_indices(rng):
        sampled_parts = []
        for class_label in sorted(groups_by_class):
            group_ids = groups_by_class[class_label]
            sampled_groups = rng.choice(
                group_ids,
                size=len(group_ids),
                replace=True,
            )
            for group_id in sampled_groups:
                sampled_parts.append(
                    group_to_indices[str(group_id)]
                )
        return np.concatenate(sampled_parts)

    return sample_indices


def percentile_interval(values, confidence_level=0.95):
    lower_probability = (1.0 - confidence_level) / 2.0
    upper_probability = 1.0 - lower_probability
    return (
        float(np.quantile(values, lower_probability)),
        float(np.quantile(values, upper_probability)),
    )


def bootstrap_two_sided_pvalue(delta_values):
    delta_values = np.asarray(delta_values, dtype=float)
    n = len(delta_values)
    probability_nonpositive = (
        np.count_nonzero(delta_values <= 0) + 1
    ) / (n + 1)
    probability_nonnegative = (
        np.count_nonzero(delta_values >= 0) + 1
    ) / (n + 1)
    return float(
        min(
            1.0,
            2.0
            * min(
                probability_nonpositive,
                probability_nonnegative,
            ),
        )
    )


def probability_matrix(bundle, status):
    if status == "native":
        columns = bundle["native_probability_columns"]
    elif status == "temperature_scaled":
        columns = bundle["calibrated_probability_columns"]
    else:
        raise ValueError(status)

    return bundle["frame"][columns].to_numpy(dtype=float)

In [ ]:
# ============================================================
# 8. Generación de índices bootstrap compartidos
# ============================================================
rng = np.random.default_rng(RANDOM_SEED)
bootstrap_indices = {}

for dataset_id in DATASETS:
    base = predictions[
        (dataset_id, REPRESENTATIONS[0], ALGORITHMS[0])
    ]["frame"]
    sampler = build_group_bootstrap_sampler(base)

    bootstrap_indices[dataset_id] = [
        sampler(rng) for _ in range(N_BOOTSTRAP)
    ]

    bootstrap_sizes = np.array(
        [len(indices) for indices in bootstrap_indices[dataset_id]]
    )
    print(
        dataset_id,
        "| réplicas:", len(bootstrap_indices[dataset_id]),
        "| tamaño mínimo:", bootstrap_sizes.min(),
        "| tamaño máximo:", bootstrap_sizes.max(),
    )

## Intervalos de confianza por modelo

Se calculan métricas puntuales e intervalos percentiles para las 32 combinaciones.  
Las métricas de discriminación son idénticas con probabilidades nativas y calibradas porque el escalamiento de temperatura preserva el `argmax`.

In [ ]:
# ============================================================
# 9. Métricas bootstrap de todos los modelos
# ============================================================

# ------------------------------------------------------------
# 9.1 Compatibilidad numérica con el notebook 05
# ------------------------------------------------------------
(
    LOG_LOSS_POLICY,
    log_loss_policy_summary,
    log_loss_policy_detailed_audit,
) = detect_log_loss_policy(
    predictions,
    reported_global_metrics,
)

log_loss_policy_summary.to_csv(
    RESULTS_DIR / "00a_log_loss_policy_summary.csv",
    index=False,
    encoding="utf-8-sig",
)
log_loss_policy_detailed_audit.to_csv(
    RESULTS_DIR / "00b_log_loss_policy_detailed_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

MAX_ALLOWED_LOG_LOSS_POLICY_DIFFERENCE = 1e-9
best_policy_max_difference = float(
    log_loss_policy_summary.iloc[0][
        "max_absolute_difference"
    ]
)

print("Política de log-loss detectada:", LOG_LOSS_POLICY)
display(log_loss_policy_summary)

if best_policy_max_difference > MAX_ALLOWED_LOG_LOSS_POLICY_DIFFERENCE:
    display(
        log_loss_policy_detailed_audit[
            log_loss_policy_detailed_audit["policy"]
            == LOG_LOSS_POLICY
        ]
        .sort_values(
            "absolute_difference",
            ascending=False,
        )
        .head(20)
    )
    raise RuntimeError(
        "No fue posible reproducir el log-loss del notebook 05 "
        "con ninguna convención numérica conocida. "
        "Revise los archivos de entrada antes de continuar."
    )


# ------------------------------------------------------------
# 9.2 Bootstrap de los 32 modelos
# ------------------------------------------------------------
point_metrics = {}
bootstrap_metric_values = {}
prepared_metric_arrays = {}
model_interval_rows = []

for dataset_id in DATASETS:
    indices_list = bootstrap_indices[dataset_id]

    for representation in REPRESENTATIONS:
        for algorithm in ALGORITHMS:
            key = (dataset_id, representation, algorithm)
            bundle = predictions[key]
            frame = bundle["frame"]
            class_order = bundle["class_order"]

            for status in ["native", "temperature_scaled"]:
                probabilities = probability_matrix(bundle, status)
                arrays = prepare_metric_arrays(
                    frame,
                    probabilities,
                    class_order,
                    LOG_LOSS_POLICY,
                )
                prepared_metric_arrays[(key, status)] = arrays
                point = fast_metrics(arrays)
                point_metrics[(key, status)] = point

                # La calibración por temperatura preserva el argmax.
                # En el estado calibrado solo se remuestrean las
                # métricas probabilísticas para evitar trabajo duplicado.
                if status == "native":
                    metrics_to_bootstrap = (
                        DISCRIMINATION_METRICS
                        + PROBABILITY_METRICS
                    )
                else:
                    metrics_to_bootstrap = PROBABILITY_METRICS

                values = {
                    metric: np.empty(
                        N_BOOTSTRAP,
                        dtype=float,
                    )
                    for metric in metrics_to_bootstrap
                }

                for bootstrap_index, sampled_indices in enumerate(
                    indices_list
                ):
                    sampled_metrics = fast_metrics(
                        arrays,
                        sampled_indices,
                    )
                    for metric in metrics_to_bootstrap:
                        values[metric][bootstrap_index] = (
                            sampled_metrics[metric]
                        )

                bootstrap_metric_values[(key, status)] = values

                for metric in metrics_to_bootstrap:
                    ci_low, ci_high = percentile_interval(
                        values[metric],
                        CONFIDENCE_LEVEL,
                    )
                    model_interval_rows.append(
                        {
                            "dataset_id": dataset_id,
                            "representation": representation,
                            "algorithm": algorithm,
                            "probability_status": status,
                            "metric": metric,
                            "estimate": point[metric],
                            "ci_low": ci_low,
                            "ci_high": ci_high,
                            "confidence_level": CONFIDENCE_LEVEL,
                            "n_bootstrap": N_BOOTSTRAP,
                            "log_loss_policy": LOG_LOSS_POLICY,
                        }
                    )

model_intervals_compact = pd.DataFrame(model_interval_rows)
model_intervals_compact.to_csv(
    TABLES_DIR / "Table_model_metric_intervals.csv",
    index=False,
    encoding="utf-8-sig",
)


# ------------------------------------------------------------
# 9.3 Auditoría exacta frente al notebook 05
# ------------------------------------------------------------
METRIC_ABSOLUTE_TOLERANCES = {
    "accuracy": 1e-12,
    "balanced_accuracy": 1e-12,
    "macro_f1": 1e-12,
    "mcc": 1e-12,
    "log_loss": 1e-9,
    "brier_multiclass": 1e-12,
}

audit_rows = []

for _, reported in reported_global_metrics.iterrows():
    key = (
        reported["dataset_id"],
        reported["representation"],
        reported["algorithm"],
    )
    status = reported["probability_status"]
    recomputed = point_metrics[(key, status)]

    for metric in (
        DISCRIMINATION_METRICS
        + PROBABILITY_METRICS
    ):
        reported_value = float(reported[metric])
        recomputed_value = float(recomputed[metric])
        difference = recomputed_value - reported_value
        tolerance = METRIC_ABSOLUTE_TOLERANCES[metric]

        audit_rows.append(
            {
                "dataset_id": key[0],
                "representation": key[1],
                "algorithm": key[2],
                "probability_status": status,
                "metric": metric,
                "reported": reported_value,
                "recomputed": recomputed_value,
                "difference": difference,
                "absolute_difference": abs(difference),
                "allowed_absolute_tolerance": tolerance,
                "passed": abs(difference) <= tolerance,
            }
        )

metric_recalculation_audit = pd.DataFrame(audit_rows)
metrics_match_notebook05 = bool(
    metric_recalculation_audit["passed"].all()
)

metric_recalculation_audit.to_csv(
    RESULTS_DIR / "00c_metric_recalculation_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

if not metrics_match_notebook05:
    failed_audit = (
        metric_recalculation_audit[
            ~metric_recalculation_audit["passed"]
        ]
        .sort_values(
            "absolute_difference",
            ascending=False,
        )
        .reset_index(drop=True)
    )
    display(failed_audit.head(30))
    raise RuntimeError(
        "Persisten diferencias materiales frente al notebook 05. "
        "El notebook se detuvo para proteger la integridad del análisis."
    )

print("Filas de intervalos:", len(model_intervals_compact))
print(
    "Máxima diferencia general frente al notebook 05:",
    metric_recalculation_audit[
        "absolute_difference"
    ].max(),
)
print(
    "Máxima diferencia de log-loss:",
    metric_recalculation_audit.loc[
        metric_recalculation_audit["metric"] == "log_loss",
        "absolute_difference",
    ].max(),
)
print("Auditoría frente al notebook 05: CORRECTA")
display(model_intervals_compact.head(20))

## Contrastes pareados entre representaciones

La diferencia se define como **representación A menos representación B**.  
Un intervalo que no cruza cero indica evidencia de diferencia en la dirección correspondiente, dentro del conjunto evaluado.

In [ ]:
# ============================================================
# 10. Contrastes pareados de representaciones
# ============================================================
contrast_rows = []

for dataset_id in DATASETS:
    for algorithm in ALGORITHMS:
        for representation_a, representation_b, contrast_name in (
            REPRESENTATION_CONTRASTS
        ):
            key_a = (dataset_id, representation_a, algorithm)
            key_b = (dataset_id, representation_b, algorithm)

            for metric in (
                DISCRIMINATION_METRICS + PROBABILITY_METRICS
            ):
                values_a = bootstrap_metric_values[
                    (key_a, "native")
                ][metric]
                values_b = bootstrap_metric_values[
                    (key_b, "native")
                ][metric]
                deltas = values_a - values_b

                point_delta = (
                    point_metrics[(key_a, "native")][metric]
                    - point_metrics[(key_b, "native")][metric]
                )
                ci_low, ci_high = percentile_interval(
                    deltas,
                    CONFIDENCE_LEVEL,
                )
                p_value = bootstrap_two_sided_pvalue(deltas)

                contrast_rows.append(
                    {
                        "dataset_id": dataset_id,
                        "algorithm": algorithm,
                        "representation_a": representation_a,
                        "representation_b": representation_b,
                        "contrast": contrast_name,
                        "metric": metric,
                        "delta_a_minus_b": point_delta,
                        "ci_low": ci_low,
                        "ci_high": ci_high,
                        "bootstrap_p_raw": p_value,
                        "confidence_level": CONFIDENCE_LEVEL,
                        "n_bootstrap": N_BOOTSTRAP,
                    }
                )

representation_contrasts = pd.DataFrame(contrast_rows)

# Corrección de Holm dentro de cada conjunto y métrica.
representation_contrasts["bootstrap_p_holm"] = np.nan
representation_contrasts["reject_holm_0_05"] = False

for (dataset_id, metric), row_indices in (
    representation_contrasts.groupby(
        ["dataset_id", "metric"]
    ).groups.items()
):
    raw_p = representation_contrasts.loc[
        row_indices, "bootstrap_p_raw"
    ].to_numpy(dtype=float)
    reject, adjusted, _, _ = multipletests(
        raw_p,
        alpha=0.05,
        method="holm",
    )
    representation_contrasts.loc[
        row_indices, "bootstrap_p_holm"
    ] = adjusted
    representation_contrasts.loc[
        row_indices, "reject_holm_0_05"
    ] = reject

representation_contrasts.to_csv(
    TABLES_DIR / "Table_paired_representation_contrasts.csv",
    index=False,
    encoding="utf-8-sig",
)

display(
    representation_contrasts[
        representation_contrasts["metric"] == "macro_f1"
    ].sort_values(
        ["dataset_id", "algorithm", "contrast"]
    )
)

In [ ]:
# ============================================================
# 11. Prueba exacta de McNemar
# ============================================================
mcnemar_rows = []

for dataset_id in DATASETS:
    for algorithm in ALGORITHMS:
        for representation_a, representation_b, contrast_name in (
            REPRESENTATION_CONTRASTS
        ):
            frame_a = predictions[
                (dataset_id, representation_a, algorithm)
            ]["frame"]
            frame_b = predictions[
                (dataset_id, representation_b, algorithm)
            ]["frame"]

            correct_a = frame_a["correct_native"].astype(bool).to_numpy()
            correct_b = frame_b["correct_native"].astype(bool).to_numpy()

            a_correct_b_wrong = int(
                np.sum(correct_a & ~correct_b)
            )
            a_wrong_b_correct = int(
                np.sum(~correct_a & correct_b)
            )
            discordant = a_correct_b_wrong + a_wrong_b_correct

            if discordant == 0:
                p_value = 1.0
            else:
                p_value = float(
                    binomtest(
                        min(a_correct_b_wrong, a_wrong_b_correct),
                        n=discordant,
                        p=0.5,
                        alternative="two-sided",
                    ).pvalue
                )

            mcnemar_rows.append(
                {
                    "dataset_id": dataset_id,
                    "algorithm": algorithm,
                    "representation_a": representation_a,
                    "representation_b": representation_b,
                    "contrast": contrast_name,
                    "a_correct_b_wrong": a_correct_b_wrong,
                    "a_wrong_b_correct": a_wrong_b_correct,
                    "discordant_total": discordant,
                    "mcnemar_exact_p_raw": p_value,
                }
            )

mcnemar_results = pd.DataFrame(mcnemar_rows)
mcnemar_results["mcnemar_exact_p_holm"] = np.nan
mcnemar_results["reject_holm_0_05"] = False

for dataset_id, row_indices in (
    mcnemar_results.groupby("dataset_id").groups.items()
):
    raw_p = mcnemar_results.loc[
        row_indices, "mcnemar_exact_p_raw"
    ].to_numpy(dtype=float)
    reject, adjusted, _, _ = multipletests(
        raw_p,
        alpha=0.05,
        method="holm",
    )
    mcnemar_results.loc[
        row_indices, "mcnemar_exact_p_holm"
    ] = adjusted
    mcnemar_results.loc[
        row_indices, "reject_holm_0_05"
    ] = reject

mcnemar_results.to_csv(
    TABLES_DIR / "Table_mcnemar_representation_contrasts.csv",
    index=False,
    encoding="utf-8-sig",
)

display(mcnemar_results)

## Efecto pareado de la calibración

La diferencia se define como:

\[
\text{métrica calibrada} - \text{métrica nativa}
\]

Para log-loss y Brier, un valor negativo representa una mejora.

In [ ]:
# ============================================================
# 12. Efecto de calibración: probabilidades calibradas - nativas
# ============================================================
calibration_rows = []

for dataset_id in DATASETS:
    for representation in REPRESENTATIONS:
        for algorithm in ALGORITHMS:
            key = (dataset_id, representation, algorithm)

            for metric in PROBABILITY_METRICS:
                native_values = bootstrap_metric_values[
                    (key, "native")
                ][metric]
                calibrated_values = bootstrap_metric_values[
                    (key, "temperature_scaled")
                ][metric]
                deltas = calibrated_values - native_values

                point_delta = (
                    point_metrics[
                        (key, "temperature_scaled")
                    ][metric]
                    - point_metrics[(key, "native")][metric]
                )
                ci_low, ci_high = percentile_interval(
                    deltas,
                    CONFIDENCE_LEVEL,
                )
                p_value = bootstrap_two_sided_pvalue(deltas)

                calibration_rows.append(
                    {
                        "dataset_id": dataset_id,
                        "representation": representation,
                        "algorithm": algorithm,
                        "metric": metric,
                        "delta_calibrated_minus_native": point_delta,
                        "ci_low": ci_low,
                        "ci_high": ci_high,
                        "bootstrap_p_raw": p_value,
                        "improved_if_negative": point_delta < 0,
                        "confidence_level": CONFIDENCE_LEVEL,
                        "n_bootstrap": N_BOOTSTRAP,
                    }
                )

calibration_effects = pd.DataFrame(calibration_rows)
calibration_effects["bootstrap_p_holm"] = np.nan
calibration_effects["reject_holm_0_05"] = False

for (dataset_id, metric), row_indices in (
    calibration_effects.groupby(
        ["dataset_id", "metric"]
    ).groups.items()
):
    raw_p = calibration_effects.loc[
        row_indices, "bootstrap_p_raw"
    ].to_numpy(dtype=float)
    reject, adjusted, _, _ = multipletests(
        raw_p,
        alpha=0.05,
        method="holm",
    )
    calibration_effects.loc[
        row_indices, "bootstrap_p_holm"
    ] = adjusted
    calibration_effects.loc[
        row_indices, "reject_holm_0_05"
    ] = reject

calibration_effects.to_csv(
    TABLES_DIR / "Table_paired_calibration_effects.csv",
    index=False,
    encoding="utf-8-sig",
)

display(
    calibration_effects.sort_values(
        [
            "dataset_id",
            "metric",
            "delta_calibrated_minus_native",
        ]
    )
)

In [ ]:
# ============================================================
# 13. Resumen de campeones seleccionados en entrenamiento
# ============================================================
champion_rows = []

for _, champion in training_champions.iterrows():
    dataset_id = champion["dataset_id"]
    representation = champion["representation"]
    algorithm = champion["algorithm"]
    key = (dataset_id, representation, algorithm)

    row = {
        "dataset_id": dataset_id,
        "representation": representation,
        "algorithm": algorithm,
        "candidate_id": champion["candidate_id"],
        "training_cv_macro_f1_mean": champion["macro_f1_mean"],
        "training_cv_macro_f1_sd": champion["macro_f1_sd"],
        "training_cv_log_loss_mean": champion["log_loss_mean"],
    }

    for metric in DISCRIMINATION_METRICS + PROBABILITY_METRICS:
        subset = model_intervals_compact[
            (model_intervals_compact["dataset_id"] == dataset_id)
            & (
                model_intervals_compact["representation"]
                == representation
            )
            & (model_intervals_compact["algorithm"] == algorithm)
            & (model_intervals_compact["metric"] == metric)
            & (
                model_intervals_compact["probability_status"]
                == "native"
            )
        ].iloc[0]

        row[f"test_{metric}"] = subset["estimate"]
        row[f"test_{metric}_ci_low"] = subset["ci_low"]
        row[f"test_{metric}_ci_high"] = subset["ci_high"]

    champion_rows.append(row)

champion_summary = pd.DataFrame(champion_rows)
champion_summary.to_csv(
    TABLES_DIR / "Table_training_selected_champions_with_test_ci.csv",
    index=False,
    encoding="utf-8-sig",
)

display(champion_summary)

## Figuras para el manuscrito

Las figuras son descriptivas e inferenciales. No se utilizan para seleccionar modelos nuevos.

In [ ]:
# ============================================================
# 14. Figuras: diferencias de macro-F1 entre representaciones
# ============================================================
def forest_plot_representation_delta(
    contrast_name,
    filename_stem,
    title,
):
    plot_data = representation_contrasts[
        (representation_contrasts["metric"] == "macro_f1")
        & (
            representation_contrasts["contrast"]
            == contrast_name
        )
    ].copy()

    plot_data["label"] = (
        plot_data["dataset_id"]
        + " | "
        + plot_data["algorithm"]
    )
    plot_data = plot_data.sort_values(
        ["dataset_id", "delta_a_minus_b"]
    ).reset_index(drop=True)

    y = np.arange(len(plot_data))
    x = plot_data["delta_a_minus_b"].to_numpy()
    lower_error = x - plot_data["ci_low"].to_numpy()
    upper_error = plot_data["ci_high"].to_numpy() - x

    figure, axis = plt.subplots(figsize=(10, 6))
    axis.errorbar(
        x,
        y,
        xerr=np.vstack([lower_error, upper_error]),
        fmt="o",
        capsize=4,
    )
    axis.axvline(0.0, linestyle="--", linewidth=1)
    axis.set_yticks(y)
    axis.set_yticklabels(plot_data["label"])
    axis.set_xlabel("Diferencia pareada de macro-F1")
    axis.set_title(title)
    axis.grid(True, axis="x", alpha=0.3)
    figure.tight_layout()

    figure.savefig(
        FIGURES_DIR / f"{filename_stem}.png",
        dpi=300,
        bbox_inches="tight",
    )
    figure.savefig(
        FIGURES_DIR / f"{filename_stem}.pdf",
        bbox_inches="tight",
    )
    plt.show()

forest_plot_representation_delta(
    "R1_minus_R3",
    "paired_macro_f1_R1_minus_R3",
    "Diferencia pareada de macro-F1: R1 − R3",
)

forest_plot_representation_delta(
    "R1_minus_R4",
    "paired_macro_f1_R1_minus_R4",
    "Diferencia pareada de macro-F1: R1 − R4",
)

forest_plot_representation_delta(
    "R3_minus_R4",
    "paired_macro_f1_R3_minus_R4",
    "Diferencia pareada de macro-F1: R3 − R4",
)

In [ ]:
# ============================================================
# 15. Figuras: efecto de calibración en log-loss
# ============================================================
for dataset_id in DATASETS:
    plot_data = calibration_effects[
        (calibration_effects["dataset_id"] == dataset_id)
        & (calibration_effects["metric"] == "log_loss")
    ].copy()

    plot_data["label"] = (
        plot_data["representation"]
        + " | "
        + plot_data["algorithm"]
    )
    plot_data = plot_data.sort_values(
        "delta_calibrated_minus_native"
    ).reset_index(drop=True)

    y = np.arange(len(plot_data))
    x = plot_data[
        "delta_calibrated_minus_native"
    ].to_numpy()
    lower_error = x - plot_data["ci_low"].to_numpy()
    upper_error = plot_data["ci_high"].to_numpy() - x

    figure, axis = plt.subplots(figsize=(11, 8))
    axis.errorbar(
        x,
        y,
        xerr=np.vstack([lower_error, upper_error]),
        fmt="o",
        capsize=4,
    )
    axis.axvline(0.0, linestyle="--", linewidth=1)
    axis.set_yticks(y)
    axis.set_yticklabels(plot_data["label"], fontsize=8)
    axis.set_xlabel(
        "Cambio de log-loss: calibrado − nativo"
    )
    axis.set_title(
        f"Efecto pareado de la calibración — {dataset_id}"
    )
    axis.grid(True, axis="x", alpha=0.3)
    figure.tight_layout()

    figure.savefig(
        FIGURES_DIR
        / f"paired_calibration_logloss_{dataset_id}.png",
        dpi=300,
        bbox_inches="tight",
    )
    figure.savefig(
        FIGURES_DIR
        / f"paired_calibration_logloss_{dataset_id}.pdf",
        bbox_inches="tight",
    )
    plt.show()

In [ ]:
# ============================================================
# 16. Resumen automático de hallazgos, sin reselección
# ============================================================
summary_rows = []

macro_contrasts = representation_contrasts[
    representation_contrasts["metric"] == "macro_f1"
].copy()

for _, row in macro_contrasts.iterrows():
    if row["ci_low"] > 0:
        direction = "A_superior"
    elif row["ci_high"] < 0:
        direction = "B_superior"
    else:
        direction = "inconcluso"

    summary_rows.append(
        {
            "analysis": "representation_macro_f1",
            "dataset_id": row["dataset_id"],
            "algorithm": row["algorithm"],
            "contrast": row["contrast"],
            "estimate": row["delta_a_minus_b"],
            "ci_low": row["ci_low"],
            "ci_high": row["ci_high"],
            "holm_significant": row["reject_holm_0_05"],
            "direction": direction,
        }
    )

for _, row in calibration_effects[
    calibration_effects["metric"] == "log_loss"
].iterrows():
    if row["ci_high"] < 0:
        direction = "calibration_improves"
    elif row["ci_low"] > 0:
        direction = "calibration_worsens"
    else:
        direction = "inconclusive"

    summary_rows.append(
        {
            "analysis": "calibration_log_loss",
            "dataset_id": row["dataset_id"],
            "algorithm": row["algorithm"],
            "contrast": row["representation"],
            "estimate": row[
                "delta_calibrated_minus_native"
            ],
            "ci_low": row["ci_low"],
            "ci_high": row["ci_high"],
            "holm_significant": row["reject_holm_0_05"],
            "direction": direction,
        }
    )

automatic_summary = pd.DataFrame(summary_rows)
automatic_summary.to_csv(
    RESULTS_DIR / "01_automatic_inferential_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Contrastes macro-F1 con ajuste de Holm:")
display(
    automatic_summary[
        (automatic_summary["analysis"] == "representation_macro_f1")
        & automatic_summary["holm_significant"]
    ]
)

print("Cambios de log-loss con ajuste de Holm:")
display(
    automatic_summary[
        (automatic_summary["analysis"] == "calibration_log_loss")
        & automatic_summary["holm_significant"]
    ]
)

In [ ]:
# ============================================================
# 17. Protocolo, validaciones finales y manifiesto SHA-256
# ============================================================
protocol = {
    "protocol_name": "Grain_AI_MDPI_paired_inference_protocol_v1",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "random_seed": RANDOM_SEED,
    "n_bootstrap": N_BOOTSTRAP,
    "confidence_level": CONFIDENCE_LEVEL,
    "log_loss_policy_detected": LOG_LOSS_POLICY,
    "log_loss_policy_max_absolute_difference": best_policy_max_difference,
    "bootstrap_design": {
        "paired": True,
        "stratified_by_class": True,
        "resampling_unit": "duplicate_group_id",
        "groups_sampled_with_replacement": True,
        "all_rows_from_sampled_group_retained": True,
    },
    "datasets": DATASETS,
    "representations": REPRESENTATIONS,
    "algorithms": ALGORITHMS,
    "prespecified_representation_contrasts": [
        {
            "representation_a": a,
            "representation_b": b,
            "contrast_name": name,
        }
        for a, b, name in REPRESENTATION_CONTRASTS
    ],
    "metrics": {
        "discrimination": DISCRIMINATION_METRICS,
        "probability": PROBABILITY_METRICS,
    },
    "tests": {
        "classification_errors": "exact McNemar test",
        "metric_differences": "paired grouped bootstrap",
        "multiple_testing": "Holm correction",
    },
    "restrictions": [
        "No model retraining.",
        "No hyperparameter reselection.",
        "No representation reselection from test results.",
        "Training-CV champions remain the primary models.",
        "No claim of image-level robustness.",
        "Inference is conditional on the frozen test datasets.",
    ],
}

(PROTOCOL_DIR / "paired_inference_protocol_v1.json").write_text(
    json.dumps(protocol, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

validation_checks = {
    "loaded_exactly_32_models": len(predictions) == 32,
    "all_prediction_sets_aligned": bool(
        alignment_audit["aligned_with_reference"].all()
    ),
    "recomputed_metrics_match_notebook05": (
        metrics_match_notebook05
    ),
    "log_loss_policy_reproduces_notebook05": (
        best_policy_max_difference
        <= MAX_ALLOWED_LOG_LOSS_POLICY_DIFFERENCE
    ),
    "no_group_has_multiple_labels": bool(
        (group_audit["groups_with_multiple_labels"] == 0).all()
    ),
    "training_champion_defined_for_each_dataset": (
        len(training_champions) == len(DATASETS)
    ),
    "all_bootstrap_arrays_complete": all(
        len(values[metric]) == N_BOOTSTRAP
        for values in bootstrap_metric_values.values()
        for metric in values
    ),
    "representation_contrasts_complete": (
        len(representation_contrasts)
        == len(DATASETS)
        * len(ALGORITHMS)
        * len(REPRESENTATION_CONTRASTS)
        * (
            len(DISCRIMINATION_METRICS)
            + len(PROBABILITY_METRICS)
        )
    ),
    "mcnemar_contrasts_complete": (
        len(mcnemar_results)
        == len(DATASETS)
        * len(ALGORITHMS)
        * len(REPRESENTATION_CONTRASTS)
    ),
    "calibration_contrasts_complete": (
        len(calibration_effects)
        == len(DATASETS)
        * len(REPRESENTATIONS)
        * len(ALGORITHMS)
        * len(PROBABILITY_METRICS)
    ),
    "no_new_champion_selected_from_test": True,
}

final_validation = pd.DataFrame(
    [
        {"check": key, "passed": value}
        for key, value in validation_checks.items()
    ]
)
final_validation.to_csv(
    RESULTS_DIR / "02_final_inference_validation.csv",
    index=False,
    encoding="utf-8-sig",
)

alignment_audit.to_csv(
    RESULTS_DIR / "03_prediction_alignment_audit.csv",
    index=False,
    encoding="utf-8-sig",
)
group_audit.to_csv(
    RESULTS_DIR / "04_group_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

if not final_validation["passed"].all():
    display(final_validation)
    raise RuntimeError("Falló al menos una validación final.")

def sha256_file(path: Path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

manifest_rows = []
for path in sorted(OUTPUT_ROOT.rglob("*")):
    if path.is_file():
        manifest_rows.append(
            {
                "relative_path": str(path.relative_to(OUTPUT_ROOT)),
                "size_bytes": path.stat().st_size,
                "sha256": sha256_file(path),
            }
        )

integrity_manifest = pd.DataFrame(manifest_rows)
integrity_manifest.to_csv(
    RESULTS_DIR / "05_output_integrity_manifest.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Validación final: CORRECTA")
display(final_validation)
print("Archivos registrados inicialmente:", len(integrity_manifest))

In [ ]:
# ============================================================
# 18. Exportación a Google Drive y creación del ZIP final
# ============================================================
DRIVE_NOTEBOOK06_ROOT = (
    DRIVE_PROJECT_ROOT / "NOTEBOOK06_PAIRED_INFERENCE_OUTPUTS"
)
if DRIVE_NOTEBOOK06_ROOT.exists():
    shutil.rmtree(DRIVE_NOTEBOOK06_ROOT)
shutil.copytree(OUTPUT_ROOT, DRIVE_NOTEBOOK06_ROOT)

FINAL_ZIP_PATH = (
    DRIVE_PROJECT_ROOT
    / "NOTEBOOK06_PAIRED_INFERENCE_OUTPUTS.zip"
)

if FINAL_ZIP_PATH.exists():
    FINAL_ZIP_PATH.unlink()

with zipfile.ZipFile(
    FINAL_ZIP_PATH,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
) as archive:
    for path in sorted(OUTPUT_ROOT.rglob("*")):
        if path.is_file():
            archive.write(
                path,
                arcname=str(path.relative_to(OUTPUT_ROOT)),
            )

print("Directorio exportado:", DRIVE_NOTEBOOK06_ROOT)
print("ZIP final:", FINAL_ZIP_PATH)
print("Tamaño ZIP (MB):", round(FINAL_ZIP_PATH.stat().st_size / 1e6, 2))

# Interpretación y uso editorial

Al terminar, el archivo principal será:

`NOTEBOOK06_PAIRED_INFERENCE_OUTPUTS.zip`

Los resultados deben interpretarse así:

- **R1 − R3 > 0:** la representación completa conserva más información discriminativa que la forma invariante.
- **R1 − R4 ≈ 0:** la representación híbrida mantiene un rendimiento comparable al conjunto completo.
- **R3 − R4 < 0:** reincorporar un ancla de tamaño mejora la clasificación.
- **Cambio calibrado − nativo < 0:** la calibración mejora log-loss o Brier.
- Un resultado no significativo no demuestra equivalencia; únicamente indica que la diferencia no quedó respaldada con este diseño y muestra.
- Los campeones definidos mediante validación cruzada en entrenamiento se mantienen como modelos principales.
- Los resultados no constituyen validación externa ni robustez frente a cambios de cámara, iluminación o segmentación.